# SPACE-GM (real package) — fast CV walkthrough

Run the genuine [`spacegm`](https://gitlab.com/enable-medicine-public/space-gm) model on the TME benchmark
with **amortized datasets**: each `spacegm.data.CellularGraphDataset` is built and processed **once** — not
rebuilt on every fold — then sliced by `train_inds` / `valid_inds`. Same patient-level cross-validation and
cohort-generalization logic as the benchmark's `cross_validate` / `cohort_split_test`, just far faster when
graph processing dominates.

Helpers live in `benchmark.models.space_gm_real_cv`:

- `cross_validate_fast(ds, task, cfg, seeds=...)` — one dataset over all CV regions, a fresh model per fold.
- `cohort_generalization_fast(ds, task, gt, cfg, seeds=...)` — two datasets (train / test cohort), sharing the
  vocabulary derived from the training cohort.

**Why it is leakage-free:** SPACE-GM featurizes each region deterministically from the explicitly-passed
vocabulary (`cell_type_mapping` / `cell_type_freq` / `biomarkers`) and fixed feature bounds — there are no
dataset-level statistics (the sanity check in section 1b confirms our featurizer reproduces
`spacegm.construct_graph_for_region` exactly). Training samples only `train_inds`
(`SubgraphSampler(selected_inds=...)`) and inference only `valid_inds`
(`collect_predict_for_all_nodes(inds=...)`), both using absolute region indices — so validation regions never
influence the fitted model even though their graphs share the dataset.

> Run this in the SPACE-GM conda env (e.g. `p3`).

## 0. Setup

In [46]:
import os
os.chdir("/autofs/bal14/zqwu/projects/TME_modeling_benchmark")
import sys, time
import numpy as np
import pandas as pd
from pathlib import Path

# REPO = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
# sys.path.insert(0, str(REPO))

import spacegm as sg
from benchmark.utils.registry import load_dataset
from benchmark.models.space_gm_real_cv import (
    SpaceGMConfig, cross_validate_fast, cohort_generalization_fast,
    derive_vocabulary, build_dataset, train_on_inds, predict_on_inds)
from benchmark.validation import summarize_folds, PRIMARY_METRIC
print('spacegm', sg.__version__)

spacegm 0.1.4


## 1. Dataset + config

A small, fast configuration for the live demo. Cell-type is the only node feature by default
(`use_center/neighbor_node_features == ['cell_type']`); set `model_dir` to persist weights.

In [2]:
dataset_name = 'crc_schurch2020'
data_root = '/autofs/bal14/zqwu/CellularTables/TME_benchmark_data'
ds = load_dataset(dataset_name, data_root=data_root)
print('tasks:', [(t, ds.get_task_config(t)['type']) for t in ds.task_ids])

cfg = SpaceGMConfig(emb_dim=512, num_iterations=2e3, batch_size=128, device='cuda:1',
                    eval_subsample_ratio=0.1)
print('features -> center:', cfg.use_center_node_features, '| neighbor:', cfg.use_neighbor_node_features)

tasks: [('CLR_DII', 'binary_classification'), ('OS', 'survival'), ('DFS', 'survival')]
features -> center: ['cell_type'] | neighbor: ['cell_type']


In [3]:
# import os, tempfile
# import numpy as np
# import pandas as pd
# import networkx as nx
# from spacegm import graph_build as gb
# from benchmark.features.space_gm_real import SpaceGMGraphBuilder

# # --- Build the SAME region two ways and compare -----------------------------
# rid = ds.get_task_metadata('primary_outcome').query("dataset == 'UPMC_HNC'")['region_id'].iloc[0]
# region = ds.load_regions([rid], normalize=False)[0]
# idx = region.coordinates.index
# mpp = float(region.microns_per_pixel)
# CUTOFF_UM = 20.0  # near-edge cutoff used by both paths

# # (A) The genuine spacegm entry point: construct_graph_for_region(graph_source='cell').
# #     It reads per-cell CSVs, so we dump this region's RAW (pixel) coords / types / expression.
# tmp = tempfile.mkdtemp(prefix='sgm_sanity_')
# coords_csv = os.path.join(tmp, 'coords.csv')
# types_csv  = os.path.join(tmp, 'types.csv')
# expr_csv   = os.path.join(tmp, 'expr.csv')
# pd.DataFrame({'CELL_ID': idx,
#               'X': region.coordinates['x'].to_numpy(),
#               'Y': region.coordinates['y'].to_numpy()}).to_csv(coords_csv, index=False)
# pd.DataFrame({'CELL_ID': idx,
#               'CELL_TYPE': region.cell_types['cell_type'].reindex(idx).to_numpy()}).to_csv(types_csv, index=False)
# expr = region.expression.reindex(idx).copy()
# expr.insert(0, 'CELL_ID', idx)
# expr.to_csv(expr_csv, index=False)

# G_real = gb.construct_graph_for_region(
#     rid,
#     cell_coords_file=coords_csv,
#     cell_types_file=types_csv,
#     cell_biomarker_expression_file=expr_csv,
#     # raw px coords -> pass physical um_per_pixel so the cutoff is evaluated in microns
#     edge_kwargs={'neighbor_edge_cutoff': CUTOFF_UM, 'um_per_pixel': mpp},
#     graph_source='cell',
# )

# # (B) Our featurizer (fit on this one region so no cell type is remapped to Unassigned).
# builder = SpaceGMGraphBuilder(near_edge_um=CUTOFF_UM).fit([region])
# G_ours = builder.transform([region]).iloc[0]['nx_graph']

# def edge_cellid_set(G):
#     """Edges as {frozenset(cell_id_a, cell_id_b)}, dropping self-loops."""
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {frozenset((cid[a], cid[b])) for a, b in G.edges() if a != b}

# def node_attr(G, key):
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {cid[n]: d.get(key) for n, d in G.nodes(data=True)}

# def edge_attr(G, key):
#     cid = nx.get_node_attributes(G, 'cell_id')
#     return {frozenset((cid[a], cid[b])): d.get(key) for a, b, d in G.edges(data=True) if a != b}

# Er, Eo = edge_cellid_set(G_real), edge_cellid_set(G_ours)
# union = Er | Eo
# print('nodes         real/ours : %d / %d' % (G_real.number_of_nodes(), G_ours.number_of_nodes()))
# print('edges         real/ours : %d / %d' % (len(Er), len(Eo)))
# print('edge topology IDENTICAL  : %s   (Jaccard %.4f)' % (Er == Eo, len(Er & Eo) / len(union)))

# # cell_type must agree cell-for-cell
# ctr, cto = node_attr(G_real, 'cell_type'), node_attr(G_ours, 'cell_type')
# ct_agree = np.mean([str(ctr[c]) == str(cto[c]) for c in ctr])
# print('cell_type agreement      : %.4f' % ct_agree)

# # edge_type (neighbor/distant) must agree on shared edges
# etr, eto = edge_attr(G_real, 'edge_type'), edge_attr(G_ours, 'edge_type')
# shared = Er & Eo
# et_agree = np.mean([etr[e] == eto[e] for e in shared])
# print('edge_type agreement      : %.4f' % et_agree)

# # --- Expected, DELIBERATE differences (not bugs) ----------------------------
# n0r, n0o = next(iter(G_real.nodes)), next(iter(G_ours.nodes))
# bm = sorted(G_ours.nodes[n0o]['biomarker_expression'])[0]
# dist_real = next(iter(edge_attr(G_real, 'distance').values()))
# dist_ours = next(iter(edge_attr(G_ours, 'distance').values()))
# bm_real = G_real.nodes[n0r]['biomarker_expression'][bm.upper()]
# bm_ours = G_ours.nodes[n0o]['biomarker_expression'][bm]
# print('\nDeliberate differences:')
# print('  voronoi_polygon  real: %s | ours: %s'
#       % (type(G_real.nodes[n0r]['voronoi_polygon']).__name__, G_ours.nodes[n0o]['voronoi_polygon']))
# print('  distance units   real: pixels  (e.g. %.2f) | ours: microns (e.g. %.2f)' % (dist_real, dist_ours))
# print('  biomarker %-8s real: raw %.3f | ours: z-scored %.3f' % (bm, bm_real, bm_ours))

## 2. Fast cross-validation (dataset processed **once**)

In [ ]:
dataset_name = 'hnc_wu2022'
data_root = '/autofs/bal14/zqwu/CellularTables/TME_benchmark_data'
ds = load_dataset(dataset_name, data_root=data_root)
print('tasks:', [(t, ds.get_task_config(t)['type']) for t in ds.task_ids])

cfg = SpaceGMConfig(emb_dim=512,
                    num_iterations=2e3,
                    batch_size=64,
                    device='cuda:0',
                    eval_subsample_ratio=0.25)
print('features -> center:', cfg.use_center_node_features, '| neighbor:', cfg.use_neighbor_node_features)

task = 'OS'
task_id = task
metric = PRIMARY_METRIC[ds.get_task_config(task)['type']]

seeds = (0, 1, 2)
model_dir = Path(f'/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{dataset_name}_{task}_cv')
work_root = Path(f'/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{dataset_name}_{task}_cv')
normalize=True

tasks: [('primary_outcome', 'binary_classification'), ('hpv_status', 'binary_classification'), ('OS', 'survival')]


In [ ]:
cross_validate_fast(
    ds, task, cfg, seeds=seeds,
    model_dir=str(model_dir), work_root=str(work_root),
)

## 3. Cohort generalization (two datasets)

The generalization test trains on UPMC and tests on DFCI, using the cohort-uniform cell types. Because the
two cohorts have different cell-type vocabularies, **two** datasets are built — but both share the mapping
derived from the *training* cohort, so the model's cell-type embeddings line up.

In [ ]:
dataset_name = 'nsclc_aung2025'
data_root = '/autofs/bal14/zqwu/CellularTables/TME_benchmark_data'
ds = load_dataset(dataset_name, data_root=data_root)
gentests = ds.validation_config['generalization_tests']

gentest = gentests[0]
print('gen test:', gentest['name'], '| train', gentest['train'], '-> test', gentest['test'],
      '| cell_type_col', gentest.get('cell_type_col'))

cfg = SpaceGMConfig(emb_dim=512,
                    num_iterations=1e3,
                    batch_size=64,
                    device='cuda:0',
                    eval_subsample_ratio=0.25)
print('features -> center:', cfg.use_center_node_features, '| neighbor:', cfg.use_neighbor_node_features)

task = 'immunotherapy_response'
task_id = task
metric = PRIMARY_METRIC[ds.get_task_config(task_id)['type']]

seeds = (0, 1, 2)
model_dir = Path(f"/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{gentest['name']}_{task}_gentest")
work_root = Path(f"/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{gentest['name']}_{task}_gentest")
normalize=True


gen test: Yale_to_YaleExt | train ['Yale'] -> test ['YaleExt'] | cell_type_col cell_type
features -> center: ['cell_type'] | neighbor: ['cell_type']


In [ ]:
cohort_generalization_fast(
    ds, task_id, gentest, cfg,
    seeds=seeds, model_dir=str(model_dir), work_root=str(work_root), normalize=True)

## 4. Special handling for multi-class classification tasks

In [43]:
dataset_name = 'bc_jackson2020'
data_root = '/autofs/bal14/zqwu/CellularTables/TME_benchmark_data'
ds = load_dataset(dataset_name, data_root=data_root)
print('tasks:', [(t, ds.get_task_config(t)['type']) for t in ds.task_ids])

cfg = SpaceGMConfig(emb_dim=512,
                    num_iterations=2e3,
                    batch_size=64,
                    device='cuda:0',
                    eval_subsample_ratio=0.25)
print('features -> center:', cfg.use_center_node_features, '| neighbor:', cfg.use_neighbor_node_features)

task = 'clinical_type'
task_id = task
metric = PRIMARY_METRIC[ds.get_task_config(task)['type']]

seeds = (0, 1, 2)
model_dir = Path(f'/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{dataset_name}_{task}_cv')
work_root = Path(f'/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{dataset_name}_{task}_cv')
normalize=True

tasks: [('OS', 'survival'), ('DFS', 'survival'), ('response', 'binary_classification'), ('clinical_type', 'multiclass_classification')]
features -> center: ['cell_type'] | neighbor: ['cell_type']


In [ ]:
cross_validate_fast(
    ds, task, cfg, seeds=seeds,
    model_dir=str(model_dir), work_root=str(work_root),
)

In [44]:
gentests = ds.validation_config['generalization_tests']
gentest = gentests[0]
print('gen test:', gentest['name'], '| train', gentest['train'], '-> test', gentest['test'],
      '| cell_type_col', gentest.get('cell_type_col'))

cfg = SpaceGMConfig(emb_dim=512,
                    num_iterations=5e3,
                    batch_size=64,
                    device='cuda:0',
                    eval_subsample_ratio=0.25)

seeds = (0, 1, 2)
model_dir = Path(f"/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{gentest['name']}_{task}_gentest")
work_root = Path(f"/autofs/bal14/zqwu/projects/TME_modeling_benchmark/{gentest['name']}_{task}_gentest")
normalize=True


gen test: Basel_to_Zurich | train ['Basel'] -> test ['Zurich'] | cell_type_col cell_type_uniform


In [ ]:
cohort_generalization_fast(
    ds, task, gentest, cfg, seeds=seeds,
    model_dir=str(model_dir), work_root=str(work_root),
)